In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=0)

agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            # model="gpt-4o-mini",
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is roleplaying a scenario where I act as the President of Lunapolis, the capital of the moon. The primary goal is to navigate the political and labor tensions involving the city's 100,000 cheese miners, specifically addressing the threat of a strike due to dissatisfaction with the current presidential administration.\n\n## SUMMARY\n\n- **Setting:** Lunapolis, the moon's capital, characterized by extreme temperature fluctuations (120C to -100C).\n- **Demographics:** The city population includes 100,000 cheese miners.\n- **Conflict:** The cheese miners' union is currently unhappy with the presidential administration, leading to a high probability of a labor strike.\n- **Strategy:** The user has requested a presidential response strategy to address the union's grievances and mitigate the strike risk.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Formulate a strategic respon

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is roleplaying a scenario where I act as the President of Lunapolis, the capital of the moon. The primary goal is to navigate the political and labor tensions involving the city's 100,000 cheese miners, specifically addressing the threat of a strike due to dissatisfaction with the current presidential administration.

## SUMMARY

- **Setting:** Lunapolis, the moon's capital, characterized by extreme temperature fluctuations (120C to -100C).
- **Demographics:** The city population includes 100,000 cheese miners.
- **Conflict:** The cheese miners' union is currently unhappy with the presidential administration, leading to a high probability of a labor strike.
- **Strategy:** The user has requested a presidential response strategy to address the union's grievances and mitigate the strike risk.

## ARTIFACTS

None.

## NEXT STEPS

- Formulate a strategic response to the cheese miners' union as the President of Luna

## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='21c4990f-19f8-4819-8e1a-65ef2636d8a8'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='e1291bf1-144a-45eb-b1af-87c2ef8a773b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='e028cc38-0d14-42dc-a037-d3f60c7b85f3'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='d25f1c5e-7c10-4855-9e8f-71a392267dd8', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='d4a3639e-a6a2-473c-9c80-e8cd96b511bf'),
              AIMessage(content=[{'type': 'text', 'text': 'I don\'t have a way to physically se

In [11]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I don\'t have a way to physically sense the temperature of your device, but **you can check it by touching the casing.**\n\nIf the device is **extremely hot to the touch**, it may have triggered a thermal safety shutdown. If that is the case:\n\n1.  **Unplug it immediately** to prevent potential damage.\n2.  **Let it sit for at least 30–60 minutes** in a cool, well-ventilated area to allow it to cool down completely.\n3.  **Check for obstructions:** Ensure that any cooling vents are not blocked by dust, blankets, or other objects.\n\n**If the device is cold or at room temperature:**\n*   **Check the power source:** Try plugging a different device (like a lamp or phone charger) into the same wall outlet to make sure the outlet itself is working.\n*   **Check the cable:** If the power cable is detachable, ensure it is pushed firmly into both the wall and the device. If you have a spare cable of the same type, try swapping it out.\n*   **Perform a "Hard Reset":*